# The actual MI300X run (outputs unedited)

This is the notebook from the AMD Developer Cloud pod that produced every number in
this repo's Results table, downloaded as-is: outputs are the real stdout of the run
on 2026-07-10. Only removed: two failed intermediate cells from before a code fix
was pulled (the fix itself is in this repo's history) and empty trailing cells.

Environment notes, honestly: the pod's egress proxy re-signs TLS, so the first cell
disables certificate verification for this throwaway session (public code and public
weights only). `torch.cuda.get_device_name` returns an empty string under this ROCm
build; the `rocm-smi` cell (device 0x744b, GPU% at 100 during training) is the
hardware proof.

**1. Environment.** Proxy workarounds, repo pull, kernel interpreter.

In [1]:
import os, sys
os.environ["CURL_CA_BUNDLE"] = ""        # pod proxy MITMs TLS; throwaway pod, public weights
os.environ["REQUESTS_CA_BUNDLE"] = ""
os.environ["SSL_CERT_FILE"] = ""
os.environ["HF_HUB_DISABLE_XET"] = "1"
%cd /workspace/apprentice-amd-hackathon
!git config --global http.sslVerify false
!git pull origin main
print(sys.executable)

/workspace/apprentice-amd-hackathon


/opt/venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.12 KiB | 1.12 MiB/s, done.
From https://github.com/singhabhishekkk/apprentice-amd-hackathon
 * branch            main       -> FETCH_HEAD
   d54715c..a155332  main       -> origin/main
Updating d54715c..a155332
error: Your local changes to the following files would be overwritten by merge:
	scripts/train_gemma_rocm.py
Please commit your changes or stash them before you merge.
Aborting
/opt/venv/bin/python3.10


**2. Hugging Face login.** Token via getpass, never in the cell or output.

In [2]:
from getpass import getpass
from huggingface_hub import login
login(token=getpass("HF token: "))

/opt/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HF token:  ········


**3. Golden dataset.** The same 200-row CUAD build as the public benchmark (seed 42).

In [3]:
import pathlib, ssl, runpy
if not pathlib.Path("data/golden.csv").exists():
    ssl._create_default_https_context = ssl._create_unverified_context
    sys.argv = ["prepare_data.py"]
    runpy.run_path("data/prepare_data.py", run_name="__main__")
    !mv golden.csv data/golden.csv
!wc -l data/golden.csv

8515 data/golden.csv


**4. Smoke.** One epoch, no baseline, proves the loop end to end (~3 min train).

In [7]:
!{sys.executable} scripts/train_gemma_rocm.py --data data/golden.csv --output-dir out/smoke --skip-baseline --epochs 1

device: AMD GPU (name unavailable in this ROCm build)
train 140, held-out 60 (seed 42)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Tokenizing train dataset: 100%|███████| 140/140 [00:00<00:00, 610.93 examples/s]
Building labels for train dataset: 100%|█| 140/140 [00:00<00:00, 2275.50 example
Truncating train dataset: 100%|██████| 140/140 [00:00<00:00, 2297.21 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.
  0%|                                                    | 0/18 [00:00<?, ?it/s]/opt/venv/lib/python3.10/site-packages/transformers/integrations/sdpa_attention.py:93: UserWarning: Using AOTriton backend for Efficient Attention forward... (Triggered internally at /app/pytorch/aten/src/ATen/native/transformers

**5. The real run.** Baseline eval, 3 epochs, final eval. This output is the Results table.

In [8]:
 !{sys.executable} scripts/train_gemma_rocm.py --data data/golden.csv --output-dir out/adapter

device: AMD GPU (name unavailable in this ROCm build)
train 140, held-out 60 (seed 42)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█████████████████████| 2076/2076 [00:05<00:00, 351.89it/s]
/opt/venv/lib/python3.10/site-packages/transformers/integrations/sdpa_attention.py:93: UserWarning: Using AOTriton backend for Efficient Attention forward... (Triggered internally at /app/pytorch/aten/src/ATen/native/transformers/hip/attention.hip:1452.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
raw (no fine-tune): 36.33
Tokenizing train dataset: 100%|███████| 140/140 [00:00<00:00, 605.22 examples/s]
Building labels for train dataset: 100%|█| 140/140 [00:00<00:00, 2242.19 example
Truncating train dataset: 100%|██████| 140/140 [00:00<00:00, 2290.43 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly,

**6. Hardware proof.** GPU% at 100 during the run.

In [9]:
 !rocm-smi



=========================================== ROCm System Management Interface ===========================================
===================================================== Concise Info =====================================================
Device  Node  IDs              Temp    Power   Partitions          SCLK     MCLK     Fan    Perf  PwrCap  VRAM%  GPU%  
              (DID,     GUID)  (Edge)  (Avg)   (Mem, Compute, ID)                                                      
0       4     0x744b,   60148  41.0°C  118.0W  N/A, N/A, 0         2227Mhz  1124Mhz  20.0%  auto  241.0W  1%     100%  
================================================= End of ROCm SMI Log ==================================================


**7. run_report.json.** The machine-readable artifact, committed alongside this notebook.

In [10]:
!cat out/adapter/run_report.json

{
  "device": "AMD GPU (name unavailable in this ROCm build)",
  "model": "google/gemma-4-E4B-it",
  "seed": 42,
  "raw_score": 36.33,
  "baseline_eval_seconds": 556.2,
  "train_seconds": 505.8,
  "finetuned_score": 61.67,
  "final_eval_seconds": 1079.8
}